In [ ]:
!wget 'https://zenodo.org/records/7119953/files/clue.zip?download=1' -O clue.zip

--2026-05-04 14:00:26--  https://zenodo.org/records/7119953/files/clue.zip?download=1
Resolving zenodo.org (zenodo.org)... 188.184.103.118, 188.185.43.153, 137.138.52.235, ...
Connecting to zenodo.org (zenodo.org)|188.184.103.118|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 635105552 (606M) [application/octet-stream]
Saving to: ‘clue.zip’

clue.zip            100%[===================>] 605.68M  33.6MB/s    in 19s     

2026-05-04 14:00:46 (31.1 MB/s) - ‘clue.zip’ saved [635105552/635105552]



In [ ]:
import os
print(os.listdir())
!unzip clue.zip

['.config', 'clue.zip', 'sample_data']
Archive:  clue.zip
  inflating: clue.json               


In [ ]:
!ls -lh clue.json

-rw-r--r-- 1 root root 14G Oct  1  2022 clue.json


In [ ]:
import json

target_bytes = 1_073_741_824  # 1GB
record_size = 297
max_records = target_bytes // record_size

print(f"Sampling {max_records:,} records...")

in_path = 'clue.json'
out_path = 'clue_sample_1GB.jsonl'

count = 0
total_bytes = 0

with open(in_path, 'r') as fin, open(out_path, 'w') as fout:
    for line in fin:
        if count >= max_records or total_bytes > target_bytes:
            break
        fout.write(line)
        count += 1
        total_bytes += len(line.encode('utf-8'))

        if count % 500000 == 0:
            print(f"  Wrote {count:,} records ({total_bytes/1024/1024:.1f} MB)")

print(f"\nDone. Wrote {count:,} records ({total_bytes/1024/1024:.1f} MB)")
print(f"File: {out_path}")

Sampling 3,615,292 records...
  Wrote 500,000 records (117.6 MB)
  Wrote 1,000,000 records (297.7 MB)
  Wrote 1,500,000 records (468.6 MB)
  Wrote 2,000,000 records (603.4 MB)
  Wrote 2,500,000 records (752.1 MB)
  Wrote 3,000,000 records (889.7 MB)

Done. Wrote 3,479,163 records (1024.0 MB)
File: clue_sample_1GB.jsonl


In [ ]:
!ls -lh clue_sample_1GB.jsonl

-rw-r--r-- 1 root root 1.1G May  4 14:03 clue_sample_1GB.jsonl


In [ ]:
with open('clue_sample_1GB.jsonl', 'r') as f:
    for i in range(3):
        print(f.readline().strip())

{"params": {"path": "/useful-turquoise-cardinal-historian"}, "type": "file_accessed", "time": "2017-07-07T08:57:57Z", "uid": "yellow-chocolate-carp-busdriver", "id": 1, "uidType": "name"}
{"params": {"path": "/useful-turquoise-cardinal-historian", "run": true}, "type": "file_updated", "time": "2017-07-07T08:58:02Z", "uid": "yellow-chocolate-carp-busdriver", "id": 2, "uidType": "name"}
{"params": {"path": "/useful-turquoise-cardinal-historian", "run": true}, "type": "file_written", "time": "2017-07-07T08:58:02Z", "uid": "yellow-chocolate-carp-busdriver", "id": 3, "uidType": "name"}


In [ ]:
import json
from collections import defaultdict, Counter

# Track users and their roles
user_roles = {}  # uid -> role
user_event_count = Counter()
user_dirs = defaultdict(set)  # uid -> set of top-level directories
user_hours = defaultdict(list)  # uid -> list of hours

print("Scanning file...")

with open('clue_sample_1GB.jsonl', 'r') as f:
    for i, line in enumerate(f):
        if i % 500000 == 0:
            print(f"  Processed {i:,} lines...")

        try:
            data = json.loads(line)
            uid = data.get('uid')
            role = data.get('role')
            time_str = data.get('time', '')
            params = data.get('params', {})
            path = params.get('path', '') if isinstance(params, dict) else ''

            if not uid:
                continue

            # Track role (first seen)
            if role and uid not in user_roles:
                user_roles[uid] = role

            # Count events per user
            user_event_count[uid] += 1

            # Track hour
            if time_str:
                hour = int(time_str[11:13]) if len(time_str) > 11 else 0
                user_hours[uid].append(hour)

            # Track top-level directory
            if path and path.startswith('/'):
                parts = path.strip('/').split('/')
                if parts:
                    user_dirs[uid].add(parts[0])

        except:
            continue

print(f"\nScan complete.")
print(f"Total users: {len(user_event_count)}")
print(f"Users with role field: {len(user_roles)}")

Scanning file...
  Processed 0 lines...
  Processed 500,000 lines...
  Processed 1,000,000 lines...
  Processed 1,500,000 lines...
  Processed 2,000,000 lines...
  Processed 2,500,000 lines...
  Processed 3,000,000 lines...

Scan complete.
Total users: 477
Users with role field: 15


In [ ]:
# Check role consistency per user
user_role_consistency = defaultdict(lambda: {'role_seen': None, 'with_role': 0, 'without_role': 0})

print("Checking role consistency...")

with open('clue_sample_1GB.jsonl', 'r') as f:
    for line in f:
        try:
            data = json.loads(line)
            uid = data.get('uid')
            role = data.get('role')

            if not uid:
                continue

            if role:
                user_role_consistency[uid]['with_role'] += 1
                if user_role_consistency[uid]['role_seen'] is None:
                    user_role_consistency[uid]['role_seen'] = role
                elif user_role_consistency[uid]['role_seen'] != role:
                    user_role_consistency[uid]['role_seen'] = 'MULTIPLE'
            else:
                user_role_consistency[uid]['without_role'] += 1

        except:
            continue

print("\n=== RESULTS ===\n")

# Question 1: Do users with role have it on ALL events?
users_with_role = {uid: data for uid, data in user_role_consistency.items()
                   if data['with_role'] > 0}

perfect_users = 0
for uid, data in users_with_role.items():
    total = data['with_role'] + data['without_role']
    if data['without_role'] == 0:
        perfect_users += 1
        print(f"{uid}: role='{data['role_seen']}' on ALL {data['with_role']} events")
    else:
        pct = (data['with_role'] / total) * 100
        print(f" {uid}: role='{data['role_seen']}' on {data['with_role']}/{total} events ({pct:.1f}%)")

print(f"\nUsers with role always present: {perfect_users}/{len(users_with_role)} ({perfect_users/len(users_with_role)*100:.1f}%)")

# Question 2: Can a user have more than one role?
users_with_multiple = [uid for uid, data in users_with_role.items()
                       if data['role_seen'] == 'MULTIPLE']

print(f"\nUsers with multiple different roles: {len(users_with_multiple)}")

if len(users_with_multiple) == 0:
    print(" Each user has a SINGLE, CONSISTENT role.")
else:
    print(f" Some users have multiple roles: {users_with_multiple}")

Checking role consistency...

=== RESULTS ===

central-silver-slug-zookeeper: role='technical' on ALL 7914 events
little-amber-minnow-reporter: role='external' on ALL 108103 events
monthly-emerald-bug-zoologist: role='technical' on ALL 3907 events
spectacular-copper-cheetah-postman: role='technical' on ALL 7473 events
inevitable-olive-meerkat-metallurgist: role='administration' on ALL 2968 events
stupid-orange-ant-tacker: role='management' on ALL 803606 events
ethnic-lavender-gerbil-gamingclubmanager: role='administration' on ALL 5593 events
lively-pink-narwhal-yachtmaster: role='technical' on ALL 2223 events
fast-coffee-ocelot-arbitrator: role='technical' on ALL 7168 events
communist-indigo-possum-forester: role='sales' on ALL 1508 events
varying-brown-nightingale-book-keeper: role='technical' on ALL 278 events
accessible-coral-ferret-repairer: role='technical' on ALL 184 events
selected-beige-vole-recorder: role='administration' on ALL 2190 events
northern-magenta-limpet-preacher: ro

In [ ]:
import pandas as pd
import numpy as np
from collections import defaultdict

# Re-scan to collect behavioral features per user
user_features = defaultdict(lambda: {
    'role': None,
    'total_events': 0,
    'unique_dirs': set(),
    'night_events': 0,
    'total_hours': set(),
    'file_events': 0,
    'login_events': 0
})

print("Collecting behavioral data for role users...")

with open('clue_sample_1GB.jsonl', 'r') as f:
    for line in f:
        try:
            data = json.loads(line)
            uid = data.get('uid')
            role = data.get('role')

            if not uid or not role:
                continue

            params = data.get('params', {})
            path = params.get('path', '') if isinstance(params, dict) else ''
            event_type = data.get('type', '')
            time_str = data.get('time', '')

            feat = user_features[uid]
            feat['role'] = role
            feat['total_events'] += 1

            if path and path.startswith('/'):
                parts = path.strip('/').split('/')
                if parts:
                    feat['unique_dirs'].add(parts[0])

            if time_str and len(time_str) > 11:
                hour = int(time_str[11:13])
                feat['total_hours'].add(hour)
                if hour <= 5:
                    feat['night_events'] += 1

            if event_type == 'file_accessed':
                feat['file_events'] += 1
            elif event_type in ['login_attempt', 'login_successful']:
                feat['login_events'] += 1

        except:
            continue

# Build comparison table
print("\n=== BEHAVIOR BY ROLE ===\n")

role_stats = defaultdict(lambda: {
    'users': [],
    'events_per_day': [],  # approximate (total/89)
    'unique_dirs': [],
    'night_frac': [],
    'active_hours': [],
    'file_ratio': []
})

for uid, feat in user_features.items():
    role = feat['role']
    days_active = len(feat['total_hours'])  # rough proxy
    events_per_day = feat['total_events'] / max(89, days_active)
    night_frac = feat['night_events'] / max(1, feat['total_events'])
    active_hours = len(feat['total_hours'])
    file_ratio = feat['file_events'] / max(1, feat['total_events'])

    role_stats[role]['users'].append(uid)
    role_stats[role]['events_per_day'].append(events_per_day)
    role_stats[role]['unique_dirs'].append(len(feat['unique_dirs']))
    role_stats[role]['night_frac'].append(night_frac)
    role_stats[role]['active_hours'].append(active_hours)
    role_stats[role]['file_ratio'].append(file_ratio)

# Print summary
for role, stats in role_stats.items():
    n = len(stats['users'])
    print(f"\n{'='*50}")
    print(f"ROLE: {role.upper()} ({n} users)")
    print(f"{'='*50}")

    print(f"\n Unique directories accessed:")
    dirs = stats['unique_dirs']
    print(f"   Min: {min(dirs)} | Max: {max(dirs)} | Mean: {np.mean(dirs):.1f} | Std: {np.std(dirs):.1f}")

    print(f"\n Events per day (approx):")
    epd = stats['events_per_day']
    print(f"   Min: {min(epd):.0f} | Max: {max(epd):.0f} | Mean: {np.mean(epd):.0f} | Std: {np.std(epd):.0f}")

    print(f"\n Night fraction (0-5 AM):")
    nf = stats['night_frac']
    print(f"   Min: {min(nf):.2%} | Max: {max(nf):.2%} | Mean: {np.mean(nf):.2%} | Std: {np.std(nf):.2%}")

    print(f"\n Active hours per day:")
    ah = stats['active_hours']
    print(f"   Min: {min(ah)} | Max: {max(ah)} | Mean: {np.mean(ah):.1f} | Std: {np.std(ah):.1f}")

    print(f"\n File access ratio (file_events / total_events):")
    fr = stats['file_ratio']
    print(f"   Min: {min(fr):.2%} | Max: {max(fr):.2%} | Mean: {np.mean(fr):.2%} | Std: {np.std(fr):.2%}")


=== BEHAVIOR BY ROLE ===


ROLE: TECHNICAL (8 users)

 Unique directories accessed:
   Min: 3 | Max: 825 | Mean: 117.8 | Std: 267.6

 Events per day (approx):
   Min: 2 | Max: 89 | Mean: 46 | Std: 33

 Night fraction (0-5 AM):
   Min: 0.42% | Max: 26.44% | Mean: 7.92% | Std: 7.73%

 Active hours per day:
   Min: 18 | Max: 24 | Mean: 22.6 | Std: 2.4

 File access ratio (file_events / total_events):
   Min: 14.39% | Max: 99.37% | Mean: 45.24% | Std: 28.35%

ROLE: EXTERNAL (1 users)

 Unique directories accessed:
   Min: 11 | Max: 11 | Mean: 11.0 | Std: 0.0

 Events per day (approx):
   Min: 1215 | Max: 1215 | Mean: 1215 | Std: 0

 Night fraction (0-5 AM):
   Min: 15.27% | Max: 15.27% | Mean: 15.27% | Std: 0.00%

 Active hours per day:
   Min: 24 | Max: 24 | Mean: 24.0 | Std: 0.0

 File access ratio (file_events / total_events):
   Min: 95.67% | Max: 95.67% | Mean: 95.67% | Std: 0.00%

ROLE: ADMINISTRATION (3 users)

 Unique directories accessed:
   Min: 2 | Max: 7 | Mean: 5.0 | Std: 2.2

In [ ]:
from collections import Counter

users = [
    "spectacular-copper-cheetah-postman",
    "ready-silver-angelfish-quarryworker"
]

event_types = Counter()

with open("clue_sample_1GB.jsonl", "r") as f:
    for line in f:
        try:
            data = json.loads(line)
            uid = data.get("uid")
            if uid in users:
                event_types[data.get("type")] += 1
        except:
            continue

print("Event types found for both users:\n")
for event, count in event_types.most_common():
    print(f"  {event}: {count:,}")

Event types found for both users:

  file_accessed: 15,631
  login_attempt: 3,639
  login_successful: 3,530
  file_written: 363
  file_created: 307
  file_deleted: 191
  deleted_from_trashbin: 95
  file_updated: 56
  file_renamed: 23
  shared_user: 2
  logout_occured: 1


# Qualitative Study of flagged events:

### Spike Mapping for consistent user


In [ ]:
import pandas as pd
import json
from collections import Counter

# ===================================================
# 1. LOAD DATA
# ===================================================
df_base = pd.read_csv("anomaly_scores_global_baseline.csv")
df_if = pd.read_csv("isolation_forest_scores.csv")
df_features = pd.read_csv("daily_features_spectacular-copper-cheetah-postman.csv")
df_features['date'] = pd.to_datetime(df_features['date']).dt.date

# Keep only flagged days
df_base = df_base[df_base['anomaly_score'] > 0]

# ===================================================
# 2. MERGE ISOLATION FOREST RESULTS
# ===================================================
if_lookup = dict(zip(df_if['date'], df_if['iso_pred'] == -1))
df_base['if_flagged'] = df_base['date'].map(if_lookup).fillna(False)
df_base['confidence'] = df_base['if_flagged'].apply(lambda x: 'HIGH' if x else 'MEDIUM')

# ===================================================
# 3. CLASSIFY ATTACK TYPE
# ===================================================
def classify_attack(top_contributors):
    if pd.isna(top_contributors) or top_contributors == "No features exceeded p95":
        return 'NONE', []

    spikes = [s.split('=')[0] for s in top_contributors.split('; ') if '=' in s]

    if 'file_events' in spikes and 'unique_paths' in spikes:
        return 'DATA_THEFT', spikes
    if 'events_total' in spikes and 'active_hours' in spikes:
        return 'BOT_OR_MASS_ACTIVITY', spikes
    if 'unique_dir1' in spikes or 'unique_dir2' in spikes:
        return 'DIRECTORY_TRAVERSAL', spikes
    if 'login_attempt' in spikes:
        return 'LOGIN_ACTIVITY', spikes
    if 'night_fraction' in spikes:
        return 'OFF_HOURS', spikes
    if 'unique_types' in spikes:
        return 'DIVERSE_ACTIVITY', spikes
    if 'events_total' in spikes and len(spikes) == 1:
        return 'MASS_ACTIVITY', spikes
    if 'path_reuse_ratio' in spikes and len(spikes) == 1:
        return 'PATH_REUSE_ANOMALY', spikes

    return 'UNKNOWN', spikes

classification = df_base['top_contributors'].apply(classify_attack)
df_base['attack_type'] = classification.apply(lambda x: x[0])
df_base['spikes'] = classification.apply(lambda x: x[1])

# ===================================================
# 4. HELPER FUNCTION FOR HOURLY DISTRIBUTION
# ===================================================
def get_hourly_distribution(uid, date):
    """Get event counts per hour from raw logs."""
    hourly = Counter()
    with open("clue_sample_1GB.jsonl", "r") as f:
        for line in f:
            try:
                data = json.loads(line)
                if data.get("uid") == uid and data.get("time", "").startswith(date):
                    hour = int(data.get("time", "")[11:13])
                    hourly[hour] += 1
            except:
                continue
    return hourly

# ===================================================
# 5. VERIFICATION FUNCTIONS
# ===================================================
def verify_data_theft(date, attack_type):
    if attack_type != 'DATA_THEFT':
        return None
    row = df_features[df_features['date'] == pd.to_datetime(date).date()]
    if row.empty:
        return 'NO_DATA'
    current = row['path_reuse_ratio'].iloc[0]
    historical = df_features[df_features['date'] < pd.to_datetime(date).date()]
    if len(historical) == 0:
        return 'INSUFFICIENT_HISTORY'
    median = historical['path_reuse_ratio'].median()
    return 'CONFIRMED' if current < median else 'SUSPICIOUS'

def verify_login(date, attack_type):
    if attack_type != 'LOGIN_ACTIVITY':
        return None
    row = df_features[df_features['date'] == pd.to_datetime(date).date()]
    if row.empty:
        return 'NO_DATA'
    rate = row['login_success_rate'].iloc[0]
    if rate < 0.5:
        return 'BRUTE_FORCE'
    elif rate > 0.8:
        return 'FALSE_POSITIVE_BUSY_DAY'
    return 'SUSPICIOUS'

def verify_bot(date, attack_type, uid):
    if attack_type != 'BOT_OR_MASS_ACTIVITY':
        return None

    row = df_features[df_features['date'] == pd.to_datetime(date).date()]
    if row.empty:
        return 'NO_DATA'

    events = row['events_total'].iloc[0]
    hours = row['active_hours'].iloc[0]

    hourly = get_hourly_distribution(uid, date)
    if not hourly:
        return 'NO_HOURLY_DATA'

    peak_hour_count = max(hourly.values())
    peak_concentration = peak_hour_count / events

    if peak_concentration > 0.5 and hours < 4:
        return 'BOT_ACTIVITY'
    else:
        return 'MASS_ACTIVITY'

# ===================================================
# 6. APPLY VERIFICATIONS
# ===================================================
df_base['data_theft_verification'] = df_base.apply(
    lambda r: verify_data_theft(r['date'], r['attack_type']), axis=1
)

df_base['login_verification'] = df_base.apply(
    lambda r: verify_login(r['date'], r['attack_type']), axis=1
)

df_base['bot_verification'] = df_base.apply(
    lambda r: verify_bot(r['date'], r['attack_type'], "spectacular-copper-cheetah-postman"), axis=1
)

# ===================================================
# 7. PRINT RESULTS
# ===================================================
print("FINAL RESULTS (Consistent User - Flagged Days Only)")
print("=" * 80)

for _, row in df_base.iterrows():
    print(f"\n{row['date']} | {row['attack_type']} | Conf: {row['confidence']} | IF: {row['if_flagged']}")

    if row['attack_type'] == 'UNKNOWN' and row['spikes']:
        print(f"    ⚠️ Spikes: {row['spikes']}")
    if row['data_theft_verification']:
        print(f"    DATA_THEFT: {row['data_theft_verification']}")
    if row['login_verification']:
        print(f"    LOGIN: {row['login_verification']}")
    if row['bot_verification']:
        print(f"    BOT/MASS: {row['bot_verification']}")

FINAL RESULTS (Consistent User - Flagged Days Only)

2017-07-17 | DATA_THEFT | Conf: HIGH | IF: True
    DATA_THEFT: CONFIRMED

2017-07-28 | DATA_THEFT | Conf: HIGH | IF: True
    DATA_THEFT: CONFIRMED

2017-08-24 | DATA_THEFT | Conf: MEDIUM | IF: False
    DATA_THEFT: SUSPICIOUS

2017-08-01 | DIRECTORY_TRAVERSAL | Conf: MEDIUM | IF: False

2017-07-15 | BOT_OR_MASS_ACTIVITY | Conf: HIGH | IF: True
    BOT/MASS: MASS_ACTIVITY

2017-09-18 | OFF_HOURS | Conf: HIGH | IF: True

2017-07-08 | LOGIN_ACTIVITY | Conf: MEDIUM | IF: False
    LOGIN: FALSE_POSITIVE_BUSY_DAY

2017-08-08 | DIVERSE_ACTIVITY | Conf: MEDIUM | IF: False

2017-07-20 | UNKNOWN | Conf: MEDIUM | IF: False
    ⚠️ Spikes: ['file_events', 'path_depth_mean']

2017-10-02 | PATH_REUSE_ANOMALY | Conf: MEDIUM | IF: False

2017-08-22 | OFF_HOURS | Conf: MEDIUM | IF: False

2017-07-16 | UNKNOWN | Conf: MEDIUM | IF: False
    ⚠️ Spikes: ['path_reuse_ratio', 'active_hours']

2017-07-14 | LOGIN_ACTIVITY | Conf: MEDIUM | IF: False
    LOG

### Spike Mapping for inconsistent user


In [ ]:
import pandas as pd
import json
from collections import Counter

# ===================================================
# 1. LOAD DATA
# ===================================================
df_base_inc = pd.read_csv("anomaly_scores_global_baseline_inconsistent.csv")
df_if_inc = pd.read_csv("isolation_forest_scores_inconsistent.csv")
df_features_inc = pd.read_csv("daily_features_ready-silver-angelfish-quarryworker.csv")
df_features_inc['date'] = pd.to_datetime(df_features_inc['date']).dt.date

# Keep only flagged days
df_base_inc = df_base_inc[df_base_inc['anomaly_score'] > 0]

# ===================================================
# 2. MERGE ISOLATION FOREST RESULTS
# ===================================================
if_lookup_inc = dict(zip(df_if_inc['date'], df_if_inc['iso_pred'] == -1))
df_base_inc['if_flagged'] = df_base_inc['date'].map(if_lookup_inc).fillna(False)
df_base_inc['confidence'] = df_base_inc['if_flagged'].apply(lambda x: 'HIGH' if x else 'MEDIUM')

# ===================================================
# 3. HELPER FUNCTION FOR HOURLY DISTRIBUTION
# ===================================================
def get_hourly_distribution(uid, date):
    """Get event counts per hour from raw logs."""
    hourly = Counter()
    with open("clue_sample_1GB.jsonl", "r") as f:
        for line in f:
            try:
                data = json.loads(line)
                if data.get("uid") == uid and data.get("time", "").startswith(date):
                    hour = int(data.get("time", "")[11:13])
                    hourly[hour] += 1
            except:
                continue
    return hourly

# ===================================================
# 4. CLASSIFY ATTACK TYPE FROM SPIKE FEATURES
# ===================================================
def classify_attack(top_contributors):
    if pd.isna(top_contributors) or top_contributors == "No features exceeded p95":
        return 'NONE', []

    spikes = [s.split('=')[0] for s in top_contributors.split('; ') if '=' in s]

    if 'file_events' in spikes and 'unique_paths' in spikes:
        return 'DATA_THEFT', spikes
    if 'events_total' in spikes and 'active_hours' in spikes:
        return 'BOT_OR_MASS_ACTIVITY', spikes
    if 'unique_dir1' in spikes or 'unique_dir2' in spikes:
        return 'DIRECTORY_TRAVERSAL', spikes
    if 'login_attempt' in spikes:
        return 'LOGIN_ACTIVITY', spikes
    if 'night_fraction' in spikes:
        return 'OFF_HOURS', spikes
    if 'unique_types' in spikes:
        return 'DIVERSE_ACTIVITY', spikes
    if 'events_total' in spikes and len(spikes) == 1:
        return 'MASS_ACTIVITY', spikes
    if 'path_reuse_ratio' in spikes and len(spikes) == 1:
        return 'PATH_REUSE_ANOMALY', spikes

    return 'UNKNOWN', spikes

classification = df_base_inc['top_contributors'].apply(classify_attack)
df_base_inc['attack_type'] = classification.apply(lambda x: x[0])
df_base_inc['spikes'] = classification.apply(lambda x: x[1])

# ===================================================
# 5. VERIFICATION FOR DATA_THEFT
# ===================================================
def verify_data_theft(date, attack_type):
    if attack_type != 'DATA_THEFT':
        return None
    row = df_features_inc[df_features_inc['date'] == pd.to_datetime(date).date()]
    if row.empty:
        return 'NO_DATA'
    current_reuse = row['path_reuse_ratio'].iloc[0]
    historical = df_features_inc[df_features_inc['date'] < pd.to_datetime(date).date()]
    if len(historical) == 0:
        return 'INSUFFICIENT_HISTORY'
    median_reuse = historical['path_reuse_ratio'].median()
    return 'CONFIRMED' if current_reuse < median_reuse else 'SUSPICIOUS'

df_base_inc['data_theft_verification'] = df_base_inc.apply(
    lambda r: verify_data_theft(r['date'], r['attack_type']), axis=1
)

# ===================================================
# 6. VERIFICATION FOR LOGIN_ACTIVITY
# ===================================================
def verify_login(date, attack_type):
    if attack_type != 'LOGIN_ACTIVITY':
        return None
    row = df_features_inc[df_features_inc['date'] == pd.to_datetime(date).date()]
    if row.empty:
        return 'NO_DATA'
    success_rate = row['login_success_rate'].iloc[0]
    if success_rate < 0.5:
        return 'BRUTE_FORCE'
    elif success_rate > 0.8:
        return 'FALSE_POSITIVE_BUSY_DAY'
    else:
        return 'SUSPICIOUS'

df_base_inc['login_verification'] = df_base_inc.apply(
    lambda r: verify_login(r['date'], r['attack_type']), axis=1
)

# ===================================================
# 7. VERIFICATION FOR BOT/MASS ACTIVITY (with hourly data)
# ===================================================
def verify_bot(date, attack_type, uid):
    if attack_type != 'BOT_OR_MASS_ACTIVITY':
        return None

    row = df_features_inc[df_features_inc['date'] == pd.to_datetime(date).date()]
    if row.empty:
        return 'NO_DATA'

    events = row['events_total'].iloc[0]
    hours = row['active_hours'].iloc[0]

    hourly = get_hourly_distribution(uid, date)
    if not hourly:
        return 'NO_HOURLY_DATA'

    peak_hour_count = max(hourly.values())
    peak_concentration = peak_hour_count / events

    # Bot: >50% of events in single hour AND active_hours < 4
    if peak_concentration > 0.5 and hours < 4:
        return 'BOT_ACTIVITY'
    else:
        return 'MASS_ACTIVITY'

df_base_inc['bot_verification'] = df_base_inc.apply(
    lambda r: verify_bot(r['date'], r['attack_type'], "ready-silver-angelfish-quarryworker"), axis=1
)

# ===================================================
# 8. PRINT RESULTS
# ===================================================
print("FINAL RESULTS (Inconsistent User - Flagged Days Only)")
print("=" * 80)

for _, row in df_base_inc.iterrows():
    print(f"\n{row['date']} | {row['attack_type']} | Conf: {row['confidence']} | IF: {row['if_flagged']}")

    if row['attack_type'] == 'UNKNOWN' and row['spikes']:
        print(f"    ⚠️ Spikes: {row['spikes']}")
    if row['data_theft_verification']:
        print(f"    DATA_THEFT: {row['data_theft_verification']}")
    if row['login_verification']:
        print(f"    LOGIN: {row['login_verification']}")
    if row['bot_verification']:
        print(f"    BOT/MASS: {row['bot_verification']}")

FINAL RESULTS (Inconsistent User - Flagged Days Only)

2017-07-11 | DATA_THEFT | Conf: HIGH | IF: True
    DATA_THEFT: CONFIRMED

2017-08-18 | DIRECTORY_TRAVERSAL | Conf: HIGH | IF: True

2017-07-23 | DIRECTORY_TRAVERSAL | Conf: HIGH | IF: True

2017-08-24 | DATA_THEFT | Conf: MEDIUM | IF: False
    DATA_THEFT: SUSPICIOUS

2017-07-17 | DIRECTORY_TRAVERSAL | Conf: HIGH | IF: True

2017-08-30 | UNKNOWN | Conf: HIGH | IF: True
    ⚠️ Spikes: ['path_reuse_ratio', 'active_hours']

2017-07-14 | DIRECTORY_TRAVERSAL | Conf: MEDIUM | IF: False

2017-07-21 | DIRECTORY_TRAVERSAL | Conf: MEDIUM | IF: False

2017-09-13 | LOGIN_ACTIVITY | Conf: MEDIUM | IF: False
    LOGIN: FALSE_POSITIVE_BUSY_DAY

2017-10-02 | OFF_HOURS | Conf: MEDIUM | IF: False

2017-08-11 | OFF_HOURS | Conf: MEDIUM | IF: False

2017-08-22 | DATA_THEFT | Conf: MEDIUM | IF: False
    DATA_THEFT: CONFIRMED

2017-07-10 | UNKNOWN | Conf: MEDIUM | IF: False
    ⚠️ Spikes: ['path_depth_mean', 'unique_paths']

2017-09-05 | UNKNOWN | Con

#### recapulative table of detection algorithm

In [ ]:
from IPython.display import HTML, display

html_table = """
<style>
.th-attack { background-color: #6a0dad; color: white; }
.th-spike { background-color: #7b1fa2; color: white; }
.th-verif { background-color: #8e24aa; color: white; }
.th-seuil { background-color: #9c27b0; color: white; }
.th-decision { background-color: #ab47bc; color: white; }
td { padding: 8px; border: 1px solid #ddd; }
tr:hover { background-color: #f5f0ff; }
</style>

<table style="border-collapse: collapse; width: 100%; font-family: Arial, sans-serif;">
<thead>
<tr style="background-color: #6a0dad; color: white;">
<th class="th-attack">Attaque</th>
<th class="th-spike">Spike(s) requis</th>
<th class="th-verif">Vérification</th>
<th class="th-seuil">Seuil</th>
<th class="th-decision">Décision</th>
</tr>
</thead>
<tbody>
<tr><td>Vol de données</td><td>file_events + unique_paths</td><td>path_reuse_ratio</td><td>&lt; médiane utilisateur</td><td style="color: #6a0dad; font-weight: bold;">DATA_THEFT</td></tr>
<tr><td>Bot</td><td>events_total + active_hours</td><td>Pic horaire > 50% ET active_hours &lt; 4</td><td>&gt; 50% et &lt; 4h</td><td style="color: #6a0dad; font-weight: bold;">BOT_ACTIVITY</td></tr>
<tr><td>Activité massive</td><td>events_total + active_hours</td><td>Pic horaire ≤ 50% OU active_hours ≥ 4</td><td>-</td><td>MASS_ACTIVITY</td></tr>
<tr><td>Brute force</td><td>login_attempt</td><td>login_success_rate</td><td>&lt; 50%</td><td style="color: #6a0dad; font-weight: bold;">BRUTE_FORCE</td></tr>
<tr><td>Faux positif login</td><td>login_attempt</td><td>login_success_rate</td><td>&gt; 80%</td><td style="color: #888; font-style: italic;">FALSE_POSITIVE_BUSY_DAY</td></tr>
<tr><td>Traversal</td><td>unique_dir1 ou unique_dir2</td><td>-</td><td>-</td><td>DIRECTORY_TRAVERSAL</td></tr>
<tr><td>Horaires décalés</td><td>night_fraction</td><td>-</td><td>-</td><td>OFF_HOURS</td></tr>
<tr><td>Non classifié</td><td>Aucune combinaison</td><td>-</td><td>-</td><td>UNKNOWN (spikes listés)</td></tr>
</tbody>
</table>
"""

display(HTML(html_table))

Attaque,Spike(s) requis,Vérification,Seuil,Décision
Vol de données,file_events + unique_paths,path_reuse_ratio,< médiane utilisateur,DATA_THEFT
Bot,events_total + active_hours,Pic horaire > 50% ET active_hours < 4,> 50% et < 4h,BOT_ACTIVITY
Activité massive,events_total + active_hours,Pic horaire ≤ 50% OU active_hours ≥ 4,-,MASS_ACTIVITY
Brute force,login_attempt,login_success_rate,< 50%,BRUTE_FORCE
Faux positif login,login_attempt,login_success_rate,> 80%,FALSE_POSITIVE_BUSY_DAY
Traversal,unique_dir1 ou unique_dir2,-,-,DIRECTORY_TRAVERSAL
Horaires décalés,night_fraction,-,-,OFF_HOURS
Non classifié,Aucune combinaison,-,-,UNKNOWN (spikes listés)


#Qualitative study of rare events

In [ ]:
from IPython.display import HTML, display

html_table = """
<style>
.th-attack { background-color: #6a0dad; color: white; }
.th-event { background-color: #7b1fa2; color: white; }
.th-window { background-color: #8e24aa; color: white; }
.th-stable { background-color: #9c27b0; color: white; }
.th-instable { background-color: #d95b5b; color: white; }
td { padding: 10px; border: 1px solid #ddd; }
tr:hover { background-color: #f5f0ff; }
.note { font-size: 0.9em; margin-top: 15px; color: #555; }
</style>

<p style="font-size: 1.1em; margin-bottom: 10px;"><strong>Phase 2 : Detection d'evenements rares et sequences</strong><br>
Seuil = (max historique par fenetre) × multiplicateur.<br>
Multiplicateur plus eleve pour l'utilisateur instable (variabilite naturelle plus grande).</p>

<table style="border-collapse: collapse; width: 100%; font-family: Arial, sans-serif;">
<thead>
<tr style="background-color: #6a0dad; color: white;">
<th class="th-attack">Attaque</th>
<th class="th-event">Evenement(s)</th>
<th class="th-window">Fenetre</th>
<th class="th-stable">Seuil (stable)</th>
<th class="th-instable">Seuil (instable)</th>
</tr>
</thead>
<tbody>
<tr style="background-color: #f9f0ff;">
<td style="font-weight: bold;">Exfiltration via partage</td>
<td><code>shared_user</code></td>
<td>instant</td>
<td>>= 1</td>
<td style="background-color: #ffe0e0;">>= 1</td>
</tr>
<tr>
<td style="font-weight: bold;">Ransomware</td>
<td><code>file_written</code></td>
<td>1 minute</td>
<td>> max × 2</td>
<td style="background-color: #ffe0e0;">>> max × 3</td>
</tr>
<tr style="background-color: #f9f0ff;">
<td style="font-weight: bold;">Suppression massive</td>
<td><code>file_deleted</code></td>
<td>5 minutes</td>
<td>> max × 2</td>
<td style="background-color: #ffe0e0;">>> max × 3</td>
</tr>
<tr>
<td style="font-weight: bold;">Depot malveillant</td>
<td><code>file_created</code></td>
<td>5 minutes</td>
<td>> max × 2</td>
<td style="background-color: #ffe0e0;">>> max × 3</td>
</tr>
<tr style="background-color: #f9f0ff;">
<td style="font-weight: bold;">Compte vole</td>
<td><code>login_successful</code> → <code>file_accessed</code></td>
<td>1 min apres login</td>
<td>> max × 2</td>
<td style="background-color: #ffe0e0;">>> max × 3</td>
</tr>
</tbody>
</table>

<p class="note"><strong>Note :</strong> Les seuils sont calcules a partir du maximum historique de chaque utilisateur. L'utilisateur instable utilise un multiplicateur plus eleve (×3) pour eviter les faux positifs lies a sa variabilite naturelle.</p>
"""

display(HTML(html_table))

Attaque,Evenement(s),Fenetre,Seuil (stable),Seuil (instable)
Exfiltration via partage,shared_user,instant,>= 1,>= 1
Ransomware,file_written,1 minute,> max × 2,>> max × 3
Suppression massive,file_deleted,5 minutes,> max × 2,>> max × 3
Depot malveillant,file_created,5 minutes,> max × 2,>> max × 3
Compte vole,login_successful → file_accessed,1 min apres login,> max × 2,>> max × 3


In [ ]:
import json
from collections import defaultdict
from datetime import datetime

def calculate_historical_max(uid):
    """Calcule le maximum historique pour chaque type d'evenement"""

    # Structures pour stocker les valeurs
    writes_1min = []      # file_written par minute
    deletes_5min = []     # file_deleted par 5 min
    creates_5min = []     # file_created par 5 min
    access_after_login = []  # accès dans la minute suivant un login

    with open("clue_sample_1GB.jsonl", "r") as f:
        # Pour suivre les compteurs par fenêtre
        current_window_1min = None
        current_writes = 0

        current_window_5min = None
        current_deletes = 0
        current_creates = 0

        last_login_time = None
        access_count = 0

        for line in f:
            try:
                data = json.loads(line)
                if data.get("uid") != uid:
                    continue

                time_str = data.get("time", "")
                if not time_str:
                    continue

                dt = datetime.fromisoformat(time_str.replace('Z', '+00:00'))
                event_type = data.get("type")

                # Fenêtre 1 minute pour file_written
                window_1min = dt.strftime("%Y-%m-%d %H:%M")
                if window_1min != current_window_1min:
                    if current_window_1min is not None:
                        writes_1min.append(current_writes)
                    current_window_1min = window_1min
                    current_writes = 0

                if event_type == "file_written":
                    current_writes += 1

                # Fenêtre 5 minutes pour file_deleted et file_created
                min_5 = (dt.minute // 5) * 5
                window_5min = dt.strftime(f"%Y-%m-%d %H:{min_5:02d}")
                if window_5min != current_window_5min:
                    if current_window_5min is not None:
                        deletes_5min.append(current_deletes)
                        creates_5min.append(current_creates)
                    current_window_5min = window_5min
                    current_deletes = 0
                    current_creates = 0

                if event_type == "file_deleted":
                    current_deletes += 1
                elif event_type == "file_created":
                    current_creates += 1

                # Accès après login
                if event_type == "login_successful":
                    last_login_time = dt
                    access_count = 0
                elif event_type == "file_accessed" and last_login_time:
                    if (dt - last_login_time).total_seconds() <= 60:
                        access_count += 1
                        access_after_login.append(access_count)

            except:
                continue

        # Ajouter les dernières fenêtres
        if current_window_1min is not None:
            writes_1min.append(current_writes)
        if current_window_5min is not None:
            deletes_5min.append(current_deletes)
            creates_5min.append(current_creates)

    return {
        "ransomware_max": max(writes_1min) if writes_1min else 0,
        "deletion_max": max(deletes_5min) if deletes_5min else 0,
        "creation_max": max(creates_5min) if creates_5min else 0,
        "account_takeover_max": max(access_after_login) if access_after_login else 0
    }

# Calculer pour les deux utilisateurs
users = {
    "spectacular-copper-cheetah-postman": calculate_historical_max("spectacular-copper-cheetah-postman"),
    "ready-silver-angelfish-quarryworker": calculate_historical_max("ready-silver-angelfish-quarryworker")
}

print("Calcul des maxima terminé")

Calcul des maxima terminé


In [ ]:
print("=" * 60)
print("SEUILS DE DETECTION PHASE 2")
print("=" * 60)

for uid, stats in users.items():
    print(f"\n{'='*40}")
    print(f"Utilisateur: {uid}")
    print(f"{'='*40}")

    if uid == "spectacular-copper-cheetah-postman":
        multi = 2  # stable
        type_user = "Stable (×2)"
    else:
        multi = 3  # instable
        type_user = "Instable (×3)"

    print(f"Profil: {type_user}")
    print(f"\nRansomware (file_written / 1 min):")
    print(f"  Max historique: {stats['ransomware_max']}")
    print(f"  Seuil (×{multi}): {stats['ransomware_max'] * multi}")

    print(f"\nSuppression massive (file_deleted / 5 min):")
    print(f"  Max historique: {stats['deletion_max']}")
    print(f"  Seuil (×{multi}): {stats['deletion_max'] * multi}")

    print(f"\nDepot malveillant (file_created / 5 min):")
    print(f"  Max historique: {stats['creation_max']}")
    print(f"  Seuil (×{multi}): {stats['creation_max'] * multi}")

    print(f"\nCompte vole (acces dans 1 min apres login):")
    print(f"  Max historique: {stats['account_takeover_max']}")
    print(f"  Seuil (×{multi}): {stats['account_takeover_max'] * multi}")

SEUILS DE DETECTION PHASE 2

Utilisateur: spectacular-copper-cheetah-postman
Profil: Stable (×2)

Ransomware (file_written / 1 min):
  Max historique: 3
  Seuil (×2): 6

Suppression massive (file_deleted / 5 min):
  Max historique: 1
  Seuil (×2): 2

Depot malveillant (file_created / 5 min):
  Max historique: 3
  Seuil (×2): 6

Compte vole (acces dans 1 min apres login):
  Max historique: 22
  Seuil (×2): 44

Utilisateur: ready-silver-angelfish-quarryworker
Profil: Instable (×3)

Ransomware (file_written / 1 min):
  Max historique: 16
  Seuil (×3): 48

Suppression massive (file_deleted / 5 min):
  Max historique: 85
  Seuil (×3): 255

Depot malveillant (file_created / 5 min):
  Max historique: 18
  Seuil (×3): 54

Compte vole (acces dans 1 min apres login):
  Max historique: 116
  Seuil (×3): 348


In [ ]:
def detect_anomalies(uid, stats, multi):
    """Detecte les fenetres qui depassent le seuil"""

    anomalies = []

    with open("clue_sample_1GB.jsonl", "r") as f:
        # Meme logique de comptage que precedemment
        # Si une fenetre depasse le seuil, on enregistre l'alerte
        # (code similaire a la cellule 1)
        pass

    return anomalies

print("Detection des anomalies (a executer apres calcul des seuils)")

Detection des anomalies (a executer apres calcul des seuils)
